In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np, pretty_midi, mir_eval
from pianotrans.pipeline import transcribe

def check(name):
    path = f"../data/test/{name}"
    ref = pretty_midi.PrettyMIDI(path + ".mid").instruments[0].notes
    est = transcribe(path + ".wav")

    ri = np.array([[n.start, n.end] for n in ref]); rp = np.array([pretty_midi.note_number_to_hz(n.pitch) for n in ref])
    ei = np.array([[n.onset, n.offset] for n in est]); ep = np.array([pretty_midi.note_number_to_hz(n.pitch) for n in est])

    on_only = mir_eval.transcription.match_notes(ri, rp, ei, ep, offset_ratio=None)
    full = mir_eval.transcription.match_notes(ri, rp, ei, ep)
    matched_ref = {i for i, _ in on_only}; matched_est = {j for _, j in on_only}
    full_ref = {i for i, _ in full}

    name_of = pretty_midi.note_number_to_name
    print(f"--- {name}: {len(ref)} true, {len(est)} detected")
    for i, n in enumerate(ref):
        if i not in matched_ref:
            print(f"  MISSED   {name_of(n.pitch):4s} {n.start:6.2f}-{n.end:6.2f}")
        elif i not in full_ref:
            j = dict(on_only)[i]
            print(f"  OFFSET   {name_of(n.pitch):4s} true end {n.end:6.2f}  detected end {est[j].offset:6.2f}")
    for j, n in enumerate(est):
        if j not in matched_est:
            print(f"  EXTRA    {name_of(n.pitch):4s} {n.onset:6.2f}-{n.offset:6.2f}")

for k in range(5):
    check(f"melody_{k:03d}")

--- melody_000: 16 true, 16 detected
  OFFSET   C#4  true end  10.37  detected end  11.38
--- melody_001: 16 true, 16 detected
  OFFSET   A4   true end   9.47  detected end  10.47
--- melody_002: 16 true, 17 detected
  OFFSET   D5   true end  12.88  detected end  13.82
  EXTRA    D5    10.21- 11.24
--- melody_003: 16 true, 16 detected
  OFFSET   B4   true end  10.55  detected end  11.56
--- melody_004: 16 true, 17 detected
  MISSED   G#4    2.88-  3.31
  OFFSET   C#4  true end   9.20  detected end  10.18
  EXTRA    G4     2.90-  3.00
  EXTRA    G#4    3.00-  3.33
